# 01 – Data Quality Assessment

## Objective

This notebook performs a structured data quality assessment of the raw credit application dataset prior to downstream analytical use.

The evaluation focuses on the following quality dimensions:

- **Completeness** – identification and quantification of missing values  
- **Consistency** – detection of inconsistent formats, categorical encodings, and schema variations  
- **Validity** – identification of impossible or out-of-range values  
- **Uniqueness** – detection of duplicate records and conflicting identifiers  

For each identified issue, we quantify its extent, assess its potential analytical impact, and apply justified remediation strategies where appropriate.

The outcome of this process is a cleaned and standardized dataset suitable for reliable fairness analysis and governance evaluation.

In [1]:
# Standard libraries
import json
import copy
from pathlib import Path

# Data manipulation
import pandas as pd
import numpy as np

# Display settings (ensuring all columns are visible)
pd.set_option("display.max_columns", None)

# Define data path
DATA_PATH = Path("../data/raw_credit_applications.json")

In [2]:
# Load raw JSON file
with open(DATA_PATH, "r") as f:
    records = json.load(f)

print(f"Total applications loaded: {len(records)}")

Total applications loaded: 502


In [3]:
# Inspect structure of first record
print("Top-level fields in first application record:")
print(records[0].keys())

Top-level fields in first application record:
dict_keys(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp'])


In [4]:
# Validate structural consistency across all records

# Collect the set of top-level keys for each record
key_sets = [set(r.keys()) for r in records]

# Identify unique structural patterns
unique_structures = {tuple(sorted(keys)) for keys in key_sets}

print(f"Number of distinct top-level structures: {len(unique_structures)}")

Number of distinct top-level structures: 4


In [5]:
# Display distinct top-level structures

for i, structure in enumerate(unique_structures, 1):
    print(f"\nStructure {i}:")
    print(structure)


Structure 1:
('_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior')

Structure 2:
('_id', 'applicant_info', 'decision', 'financials', 'spending_behavior')

Structure 3:
('_id', 'applicant_info', 'decision', 'financials', 'loan_purpose', 'spending_behavior')

Structure 4:
('_id', 'applicant_info', 'decision', 'financials', 'processing_timestamp', 'spending_behavior')


### Top-Level Schema Variation

The dataset exhibits four distinct top-level schema variations.

A core schema structure is consistently present across all records, comprising:
- `_id`
- `applicant_info`
- `financials`
- `decision`
- `spending_behavior`

However, the following additional fields appear inconsistently:
- `loan_purpose`
- `processing_timestamp`
- `notes`

This indicates that certain attributes are optional and inconsistently populated across the dataset. Prior to transformation and analysis, the schema must be unified to ensure consistent representation of all expected fields.

In [6]:
# Quantify presence of optional top-level fields

optional_fields = ["loan_purpose", "processing_timestamp", "notes"]

total_records = len(records)

summary = []

for field in optional_fields:
    present = sum(field in r for r in records)
    summary.append({
        "field": field,
        "present_count": present,
        "present_pct (%)": round((present / total_records) * 100, 2)
    })

pd.DataFrame(summary).sort_values("present_count", ascending=False).reset_index(drop=True)

,field,present_count,present_pct (%)
0,processing_timestamp,62,12.35
1,loan_purpose,50,9.96
2,notes,2,0.40


### Schema Normalization

The dataset contains a consistent core structure, but several top-level fields appear only in subsets of records (e.g., `loan_purpose`, `processing_timestamp`, `notes`). 

To ensure consistent processing and reproducible transformation steps, we standardize the top-level schema by explicitly adding these optional fields to all records when absent and representing missing values as null. This does not imply that these fields are mandatory. It enforces a uniform schema representation prior to flattening and downstream analysis.

In [7]:
records_norm = copy.deepcopy(records)  # work on a copy; keep `records` as raw reference

In [8]:
# Define a canonical top-level schema for consistent processing

canonical_fields = [
    "_id",
    "applicant_info",
    "financials",
    "decision",
    "spending_behavior",
    "loan_purpose",
    "processing_timestamp",
    "notes",
]

In [9]:
# Normalize records_norm: ensure all canonical fields exist in every record
# Missing optional fields are added as explicit nulls (None)

for r in records_norm:
    for field in canonical_fields:
        if field not in r:
            r[field] = None

In [10]:
# Verify that all records now share the same top-level structure after normalization
normalized_structures = {tuple(sorted(r.keys())) for r in records_norm}

assert len(normalized_structures) == 1, "Schema normalization failed"
print(f"All {len(records_norm)} records now share a unified top-level schema.")

All 502 records now share a unified top-level schema.


In [11]:
df_discovery = pd.json_normalize(records_norm, sep=".")
df_discovery.columns = [c.replace(".", "_") for c in df_discovery.columns]

print("df_discovery shape:", df_discovery.shape)
df_discovery.head()

df_discovery shape: (502, 21)


,_id,spending_behavior,processing_timestamp,loan_purpose,notes,applicant_info_full_name,applicant_info_email,applicant_info_ssn,applicant_info_ip_address,applicant_info_gender,applicant_info_date_of_birth,applicant_info_zip_code,financials_annual_income,financials_credit_history_months,financials_debt_to_income,financials_savings_balance,decision_loan_approved,decision_rejection_reason,decision_interest_rate,decision_approved_amount,financials_annual_salary
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,None,None,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",None,None,None,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",None,vacation,None,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,3.7,59000.0,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",None,None,None,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,0.35,0,True,NaN,4.3,34000.0,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,None,None,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN


### Schema drift check (core fields)

Before cleaning, we check whether the same concept appears under different field names (e.g., income recorded as `annual_income` vs `annual_salary`). If so, we will standardize it to avoid treating valid values as missing.

In [12]:
# Schema overview: how many columns per nested section?
app_cols = [c for c in df_discovery.columns if c.startswith("applicant_info_")]
fin_cols = [c for c in df_discovery.columns if c.startswith("financials_")]
dec_cols = [c for c in df_discovery.columns if c.startswith("decision_")]

print(f"Applicant fields: {len(app_cols)}")
print(f"Financial fields: {len(fin_cols)}")
print(f"Decision fields: {len(dec_cols)}")

# Focused drift check: income fields (annual_income vs annual_salary)
income_like = [c for c in fin_cols if ("annual_income" in c) or ("annual_salary" in c)]
print("\nIncome-related financial fields detected:", income_like)

# Quantify presence (non-null) to show drift impact
for col in income_like:
    non_null = df_discovery[col].notna().sum()
    print(f"  {col}: non-null in {non_null} records ({(non_null/len(df_discovery))*100:.2f}%)")

Applicant fields: 7
Financial fields: 5
Decision fields: 4

Income-related financial fields detected: ['financials_annual_income', 'financials_annual_salary']
  financials_annual_income: non-null in 497 records (99.00%)
  financials_annual_salary: non-null in 5 records (1.00%)


Income is recorded under two alternative fields. `financials_annual_income` is populated for 497/502 applications, while `financials_annual_salary` is populated for 5/502. These represent the same concept and will be merged into a single canonical `financials_annual_income_canonical` field in the curated dataset.

In [13]:
# Structural fix: merge annual_income and annual_salary into one canonical field

df_work = df_discovery.copy()

income_col = "financials_annual_income"
salary_col = "financials_annual_salary"

income_num = pd.to_numeric(df_work[income_col], errors="coerce")
salary_num = pd.to_numeric(df_work[salary_col], errors="coerce")

df_work["financials_annual_income_canonical"] = income_num.combine_first(salary_num)

# Structural verification
missing_after_merge = df_work["financials_annual_income_canonical"].isna().sum()

print(
    "Missing canonical income after structural merge:",
    missing_after_merge,
    "/",
    len(df_work)
)

Missing canonical income after structural merge: 0 / 502


In [14]:
# Exclude two columns from DQ analysis - replaced by canonical income
exclude_cols = [
    "financials_annual_income",
    "financials_annual_salary"
]

## Completeness

After resolving structural inconsistencies,  we assess missing values across key column groups. The objective is to quantify incomplete records before applying any remediation.

In [15]:
# Column grouping by schema section
core_cols      = [c for c in df_work.columns if c in ["_id", "processing_timestamp", "loan_purpose", "notes"]]
applicant_info_cols = [c for c in df_work.columns if c.startswith("applicant_info_")]
financial_cols = [
    c for c in df_work.columns
    if c.startswith("financials_")
    and c not in exclude_cols
]
spending_cols  = [c for c in df_work.columns if c.startswith("spending_")]
decision_cols  = [c for c in df_work.columns if c.startswith("decision_")]

column_groups = {
    "core":      core_cols,
    "applicant": applicant_info_cols,
    "financial": financial_cols,
    "spending":  spending_cols,
    "decision":  decision_cols,
}

# Fields where missingness is structurally expected:
# - Decision fields: rejected apps have no interest rate or approved amount;
#                   approved apps have no rejection reason
# - Optional fields: documented during schema inspection as inconsistently
#                   populated by design, not data quality failures
EXPECTED_MISSING = {
    "decision_interest_rate",
    "decision_approved_amount",
    "decision_rejection_reason",
    "notes",                     # optional — present in 0.40% of records
    "loan_purpose",              # optional — present in 9.96% of records
    "processing_timestamp",      # optional — present in 12.35% of records
}

# Completeness calculation
n_records = len(df_work)
rows = []

for group_name, cols in column_groups.items():
    for col in cols:
        missing_count = df_work[col].isna().sum()
        rows.append({
            "column":        col,
            "group":         group_name,
            "missing_count": missing_count,
            "missing_pct":   round(missing_count / n_records * 100, 2),
            "expected":      col in EXPECTED_MISSING,
        })

completeness_df = (
    pd.DataFrame(rows)
    .sort_values(["missing_count", "group"], ascending=[False, True])
    .reset_index(drop=True)
)

completeness_df

,column,group,missing_count,missing_pct,expected
0,notes,core,500,99.60,True
1,loan_purpose,core,452,90.04,True
2,processing_timestamp,core,440,87.65,True
3,decision_rejection_reason,decision,292,58.17,True
4,decision_interest_rate,decision,210,41.83,True
5,decision_approved_amount,decision,210,41.83,True
6,applicant_info_ssn,applicant,5,1.00,False
7,applicant_info_ip_address,applicant,5,1.00,False
8,applicant_info_gender,applicant,1,0.20,False
9,applicant_info_date_of_birth,applicant,1,0.20,False


High missingness is mainly concentrated in optional fields such as `notes`, `loan_purpose`, and `processing_timestamp`, so it is treated as expected.

In [16]:
# Identify columns with unexpected missing values
genuine_issues = completeness_df[
    (completeness_df["missing_count"] > 0) & 
    (completeness_df["expected"] == False)
]

print(f"Columns with unexpected missing values: {len(genuine_issues)}")
genuine_issues

Columns with unexpected missing values: 5


,column,group,missing_count,missing_pct,expected
6,applicant_info_ssn,applicant,5,1.0,False
7,applicant_info_ip_address,applicant,5,1.0,False
8,applicant_info_gender,applicant,1,0.2,False
9,applicant_info_date_of_birth,applicant,1,0.2,False
10,applicant_info_zip_code,applicant,1,0.2,False


The table above shows missing values in key applicant identifiers and demographic fields (SSN, IP address, gender, date of birth, and ZIP code). Even though the counts are low, these attributes are important for traceability and governance. The affected records will be flagged and handled during remediation to ensure the curated dataset remains reliable.

## Consistency

After the completeness check, we look for logical inconsistencies between related fields. In particular, decision attributes (interest rate, approved 
amount, rejection reason) should align with the loan approval outcome. We also review basic consistency in key categorical fields (e.g., gender coding 
and date formats) before remediation.

In [17]:
# Consistency: Decision logic checks

n_rows = len(df_work)

required = [
    "decision_loan_approved",
    "decision_interest_rate",
    "decision_approved_amount",
    "decision_rejection_reason",
]
missing_cols = [c for c in required if c not in df_work.columns]
if missing_cols:
    print("Missing expected decision columns:", missing_cols)

approved = df_work["decision_loan_approved"]

has_rate = df_work["decision_interest_rate"].notna()
has_amount = df_work["decision_approved_amount"].notna()
has_rejection = df_work["decision_rejection_reason"].notna()

# Rules
rej_has_terms = (approved == False) & (has_rate | has_amount)
app_has_rejection = (approved == True) & has_rejection
app_missing_terms = (approved == True) & (~has_rate | ~has_amount)

print(f"Rejected but has interest/amount: {rej_has_terms.sum()}")
print(f"Approved but has rejection reason: {app_has_rejection.sum()}")
print(f"Approved but missing interest/amount: {app_missing_terms.sum()}")

# Store for later remediation section
df_work["dq_flag_decision_rejected_has_terms"] = rej_has_terms
df_work["dq_flag_decision_approved_has_rejection"] = app_has_rejection
df_work["dq_flag_decision_approved_missing_terms"] = app_missing_terms

Rejected but has interest/amount: 0
Approved but has rejection reason: 0
Approved but missing interest/amount: 0


All decision fields are logically consistent with the approval outcome. Rejected applications do not contain loan terms (interest rate / approved amount), approved applications include the required terms, and rejection reasons only appear for rejected cases.

In [18]:
df_work["applicant_info_gender"].value_counts(dropna=False)

applicant_info_gender
Male      195
Female    193
F          58
M          53
            2
NaN         1
Name: count, dtype: int64

Gender values are not consistently coded. The dataset contains both full labels (`Male`, `Female`) and abbreviations (`M`, `F`), as well as a small number of blank/missing entries. We will standardize these to a single representation during remediation.

In [19]:
# Consistency: ZIP code format

zip_col = "applicant_info_zip_code"

zip_raw = df_work[zip_col].astype("string").str.strip()

# Treat empty strings as missing
zip_blank = df_work[zip_col].notna() & (zip_raw == "")

# Only evaluate non-blank present ZIPs
zip_present = df_work[zip_col].notna() & (zip_raw != "")

# Keep digits only (handles spaces, dashes, etc.)
zip_digits = zip_raw.str.replace(r"\D", "", regex=True)
zip_len = zip_digits.str.len()

# Rule: if ZIP is present, it should contain exactly 5 digits
zip_bad = zip_present & (zip_len != 5)

# Placeholder case
zip_placeholder_zero = zip_present & (zip_digits == "0")

print("ZIP blank strings:", int(zip_blank.sum()))
print("ZIP present but not 5 digits:", int(zip_bad.sum()))
print("ZIP placeholder '0' detected:", int(zip_placeholder_zero.sum()))

# Store flags for remediation
df_work["dq_flag_zip_blank_string"] = zip_blank
df_work["dq_flag_zip_bad_format"] = zip_bad
df_work["dq_flag_zip_placeholder_zero"] = zip_placeholder_zero

# Show details only if there are format issues
zip_issue_mask = zip_bad | zip_placeholder_zero
if zip_issue_mask.any():
    zip_issues_preview = df_work.loc[zip_issue_mask, ["_id", zip_col]].copy()
    zip_issues_preview["zip_digits_only"] = zip_digits.loc[zip_issue_mask].values
    zip_issues_preview["zip_digit_len"] = zip_len.loc[zip_issue_mask].values
    display(zip_issues_preview.head(10))
else:
    print("No ZIP format issues to display (only blank-string missingness).")
    
if zip_blank.any():
    print("Example blank ZIP rows:")
    display(df_work.loc[zip_blank, ["_id", zip_col]].head(5))

ZIP blank strings: 1
ZIP present but not 5 digits: 0
ZIP placeholder '0' detected: 0
No ZIP format issues to display (only blank-string missingness).
Example blank ZIP rows:


,_id,applicant_info_zip_code
26,app_075,


ZIP values are correctly formatted when present. One record (`app_075`)uses an empty string instead of a missing value. This will be standardized to `NaN` during remediation.

In [20]:
# Consistency: IP address format

ip_col = "applicant_info_ip_address"

ip_present = df_work[ip_col].notna()
ip_raw = df_work[ip_col].astype("string").str.strip()

# Basic IPv4 pattern: ddd.ddd.ddd.ddd
ip_bad = ip_present & (~ip_raw.str.match(r"^(\d{1,3}\.){3}\d{1,3}$", na=False))

print("IPs present but invalid IPv4 format:", int(ip_bad.sum()))

# Store flag for remediation
df_work["dq_flag_ip_bad_format"] = ip_bad

# Show examples only if any
if ip_bad.any():
    display(df_work.loc[ip_bad, ["_id", ip_col]].head(10))

IPs present but invalid IPv4 format: 0


All present IP addresses follow valid IPv4 format. No consistency issues detected.

In [21]:
# Consistency: SSN format — mask output

ssn_col = "applicant_info_ssn"

ssn_present = df_work[ssn_col].notna()
ssn_raw = df_work[ssn_col].astype("string").str.strip()

ssn_digits = ssn_raw.str.replace(r"\D", "", regex=True)
ssn_bad = ssn_present & (ssn_digits.str.len() != 9)

print("SSNs present but not 9 digits:", int(ssn_bad.sum()))

# Store flag for remediation
df_work["dq_flag_ssn_bad_format"] = ssn_bad

# Show masked examples only if any
if ssn_bad.any():
    tmp = df_work.loc[ssn_bad, ["_id"]].copy()
    tmp["ssn_masked"] = "***-**-" + ssn_digits.loc[ssn_bad].str[-4:]
    display(tmp.head(10))

SSNs present but not 9 digits: 0


SSNs are consistently formatted (all present values normalize to 9 digits).

We assess format consistency in `date_of_birth` by identifying whether values follow a single standardized representation (ISO``YYYY-MM-DD``) or appear in mixed formats. This check is important because inconsistent date formats can affect downstream parsing, age calculations, and duplicate detection.

In [43]:
# Consistency - Date format consistency

import re

dob_col = "applicant_info_date_of_birth"
n_rows = len(df_work)

def pct(x, total=n_rows):
    return round((x / total) * 100, 2) if total else 0.0

dob_raw = df_work[dob_col].astype("string")
dob_stripped = dob_raw.str.strip()

# Separate completeness-related cases from format consistency
dob_missing_mask = df_work[dob_col].isna()
dob_blank_mask = df_work[dob_col].notna() & (dob_stripped == "")

def detect_dob_format(v):
    if pd.isna(v):
        return "missing"
    s = str(v).strip()
    if s == "":
        return "blank"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return "YYYY-MM-DD (ISO)"
    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", s):
        return "YYYY/MM/DD"
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", s):
        return "DD/MM/YYYY or MM/DD/YYYY"
    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", s):
        return "DD-MM-YYYY or MM-DD-YYYY"
    return "other/unrecognized"

df_work["analysis_dob_format"] = df_work[dob_col].apply(detect_dob_format)

dob_format_dist = (
    df_work["analysis_dob_format"]
    .value_counts(dropna=False)
    .rename_axis("dob_format_category")
    .reset_index(name="count")
)
dob_format_dist["pct"] = (dob_format_dist["count"] / n_rows * 100).round(2)

display(dob_format_dist)

,dob_format_category,count,pct
0,YYYY-MM-DD (ISO),340,67.73
1,DD/MM/YYYY or MM/DD/YYYY,101,20.12
2,YYYY/MM/DD,56,11.16
3,blank,4,0.80
4,missing,1,0.20


In [46]:
# Create  helper column for downstream analysis (e.g., age validity checks)

from datetime import datetime

def parse_dob_multi(v):
    if pd.isna(v):
        return pd.NaT
    s = str(v).strip()
    if s == "":
        return pd.NaT
    
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%d-%m-%Y", "%m/%d/%Y", "%m-%d-%Y"):
        try:
            return pd.Timestamp(datetime.strptime(s, fmt))
        except ValueError:
            continue
    return pd.NaT

df_work["analysis_dob_parsed"] = df_work[dob_col].apply(parse_dob_multi)

# Flags for later reporting/remediation
dob_unparseable_mask = (
    ~dob_missing_mask
    & ~dob_blank_mask
    & df_work["analysis_dob_parsed"].isna()
)

dob_non_iso_valid_mask = (
    ~dob_missing_mask
    & ~dob_blank_mask
    & (df_work["analysis_dob_format"] != "YYYY-MM-DD (ISO)")
    & (~df_work["analysis_dob_parsed"].isna())
)

df_work["dq_flag_dob_blank_string"] = dob_blank_mask
df_work["dq_flag_dob_non_iso_format"] = dob_non_iso_valid_mask
df_work["dq_flag_dob_unparseable"] = dob_unparseable_mask

print(f"DOB missing values: {int(dob_missing_mask.sum())} ({pct(int(dob_missing_mask.sum()))}%)")
print(f"DOB blank strings: {int(dob_blank_mask.sum())} ({pct(int(dob_blank_mask.sum()))}%)")
print(f"DOB valid non-ISO formats: {int(dob_non_iso_valid_mask.sum())} ({pct(int(dob_non_iso_valid_mask.sum()))}%)")
print(f"DOB unparseable values: {int(dob_unparseable_mask.sum())} ({pct(int(dob_unparseable_mask.sum()))}%)")

if dob_unparseable_mask.any():
    display(
        df_work.loc[dob_unparseable_mask, ["_id", "applicant_info_full_name", dob_col]]
        .head(10)
    )

DOB missing values: 1 (0.2%)
DOB blank strings: 4 (0.8%)
DOB valid non-ISO formats: 157 (31.27%)
DOB unparseable values: 0 (0.0%)


In [49]:
# Data quality summary by dimension (count + percentage)
dob_consistency_summary = pd.DataFrame([
    {
        "issue": "DOB missing values",
        "dq_dimension": "Completeness",
        "affected_rows": int(dob_missing_mask.sum())
    },
    {
        "issue": "DOB blank strings",
        "dq_dimension": "Completeness",
        "affected_rows": int(df_work["dq_flag_dob_blank_string"].sum())
    },
    {
        "issue": "DOB valid non-ISO formats",
        "dq_dimension": "Consistency",
        "affected_rows": int(df_work["dq_flag_dob_non_iso_format"].sum())
    },
    {
        "issue": "DOB unparseable values",
        "dq_dimension": "Validity",
        "affected_rows": int(df_work["dq_flag_dob_unparseable"].sum())
    },
])

dob_consistency_summary["affected_pct"] = (dob_consistency_summary["affected_rows"] / n_rows * 100).round(2)
display(dob_consistency_summary)

,issue,dq_dimension,affected_rows,affected_pct
0,DOB missing values,Completeness,1,0.20
1,DOB blank strings,Completeness,4,0.80
2,DOB valid non-ISO formats,Consistency,157,31.27
3,DOB unparseable values,Validity,0,0.00


The `date_of_birth` assessment shows that the main issue is format inconsistency rather than invalid date values. While a small number of records contain missing or blank DOB entries (completeness issues), a substantial share of records (**31.27%**) store valid DOB values in non-standard (non-ISO) formats (consistency issue). No unparseable DOB values were identified in this dataset. This supports a remediation strategy focused primarily on date standardization rather than record exclusion.

## Uniqueness

We next assess **uniqueness** by checking whether records that should be unique are repeated in the dataset.

The main checks focus on:
- duplicate application identifiers (`_id`)
- duplicate SSNs (excluding missing values)
- duplicate combinations of applicant name and date of birth (possible repeated applications or duplicate records)

This section only performs detection and quantification. Any deduplication decisions are deferred to the Remediation section to avoid changing the dataset during the assessment stage.

In [22]:
# Uniqueness - duplicate detection

n_rows = len(df_work)

def pct(x, total=n_rows):
    return round((x / total) * 100, 2) if total else 0.0

# Duplicate application IDs (_id)
id_series = df_work["_id"].astype("string").str.strip()
id_present_mask = id_series.notna() & (id_series != "")
dup_id_mask = id_present_mask & id_series.duplicated(keep=False)

print(f"Rows with duplicate _id: {int(dup_id_mask.sum())} ({pct(int(dup_id_mask.sum()))}%)")

if dup_id_mask.any():
    display(
        df_work.loc[dup_id_mask, ["_id", "applicant_info_full_name", "applicant_info_date_of_birth"]]
        .sort_values("_id")
    )

# Save flag for later remediation
df_work["dq_flag_duplicate_id"] = dup_id_mask

Rows with duplicate _id: 4 (0.8%)


,_id,applicant_info_full_name,applicant_info_date_of_birth
383,app_001,Stephanie Nguyen,1986-05-27
455,app_001,Stephanie Nguyen,NaN
8,app_042,Joseph Lopez,1990-05-04
354,app_042,Joseph Lopez,1990-05-04


In [23]:
# Duplicate SSNs (SAFE detection: exclude missing/blank values)
ssn_col = "applicant_info_ssn"

ssn_series = df_work[ssn_col].astype("string").str.strip()
ssn_present_mask = ssn_series.notna() & (ssn_series != "")

# Normalize SSN for matching (remove separators)
ssn_norm = ssn_series.str.replace(r"\D", "", regex=True)

# Only treat 9-digit SSNs as valid duplicate keys
ssn_valid_mask = ssn_present_mask & (ssn_norm.str.len() == 9)

dup_ssn_mask = ssn_valid_mask & ssn_norm.duplicated(keep=False)

print(f"Rows with duplicate valid SSN: {int(dup_ssn_mask.sum())} ({pct(int(dup_ssn_mask.sum()))}%)")

if dup_ssn_mask.any():
    ssn_preview = df_work.loc[
        dup_ssn_mask,
        ["_id", "applicant_info_full_name", "applicant_info_date_of_birth"]
    ].copy()
    ssn_preview["ssn_masked"] = "***-**-" + ssn_norm.loc[dup_ssn_mask].str[-4:].values
    display(ssn_preview.sort_values(["ssn_masked", "_id"]))

# Save flag for later remediation
df_work["dq_flag_duplicate_ssn"] = dup_ssn_mask

Rows with duplicate valid SSN: 6 (1.2%)


,_id,applicant_info_full_name,applicant_info_date_of_birth,ssn_masked
8,app_042,Joseph Lopez,1990-05-04,***-**-5530
354,app_042,Joseph Lopez,1990-05-04,***-**-5530
16,app_101,Sandra Smith,1997-03-23,***-**-8731
499,app_234,Samuel Hill,1976/01/29,***-**-8731
122,app_016,Gary Wilson,1959-12-11,***-**-9300
92,app_088,Susan Martinez,1986-10-15,***-**-9300


In [26]:
# Duplicate Full Name + DOB
name_series = df_work["applicant_info_full_name"].astype("string").str.strip().str.lower()
dob_series = df_work["applicant_info_date_of_birth"].astype("string").str.strip()

name_present_mask = name_series.notna() & (name_series != "")
dob_present_mask = dob_series.notna() & (dob_series != "")
key_present_mask = name_present_mask & dob_present_mask

name_dob_key = pd.Series(list(zip(name_series, dob_series)), index=df_work.index)
dup_name_dob_mask = key_present_mask & name_dob_key.duplicated(keep=False)

print(f"Rows with duplicate Full Name + DOB (raw string): {int(dup_name_dob_mask.sum())} ({pct(int(dup_name_dob_mask.sum()))}%)")

if dup_name_dob_mask.any():
    name_dob_preview = df_work.loc[
        dup_name_dob_mask,
        ["_id", "applicant_info_full_name", "applicant_info_date_of_birth", "applicant_info_ssn"]
    ].copy()

    # Mask SSN for display (keep only last 4 digits)
    ssn_preview_raw = name_dob_preview["applicant_info_ssn"].astype("string").str.strip()
    ssn_preview_digits = ssn_preview_raw.str.replace(r"\D", "", regex=True)

    name_dob_preview["applicant_info_ssn_masked"] = np.where(
        ssn_preview_digits.str.len() == 9,
        "***-**-" + ssn_preview_digits.str[-4:],
        pd.NA
    )

    # Drop raw SSN from the preview to avoid exposing PII
    name_dob_preview = name_dob_preview.drop(columns=["applicant_info_ssn"])

    display(
        name_dob_preview.sort_values(
            ["applicant_info_full_name", "applicant_info_date_of_birth"]
        )
    )

# Save flag for later remediation
df_work["dq_flag_duplicate_name_dob"] = dup_name_dob_mask

Rows with duplicate Full Name + DOB (raw string): 2 (0.4%)


,_id,applicant_info_full_name,applicant_info_date_of_birth,applicant_info_ssn_masked
8,app_042,Joseph Lopez,1990-05-04,***-**-5530
354,app_042,Joseph Lopez,1990-05-04,***-**-5530


In [27]:
# Uniqueness summary (count + percentage)
uniqueness_summary = pd.DataFrame([
    {
        "issue": "Duplicate application ID (_id)",
        "dq_dimension": "Uniqueness",
        "affected_rows": int(df_work["dq_flag_duplicate_id"].sum())
    },
    {
        "issue": "Duplicate valid SSN",
        "dq_dimension": "Uniqueness",
        "affected_rows": int(df_work["dq_flag_duplicate_ssn"].sum())
    },
    {
        "issue": "Duplicate Full Name + DOB (raw DOB string)",
        "dq_dimension": "Uniqueness",
        "affected_rows": int(df_work["dq_flag_duplicate_name_dob"].sum())
    }
])

uniqueness_summary["affected_pct"] = (uniqueness_summary["affected_rows"] / n_rows * 100).round(2)
display(uniqueness_summary)

,issue,dq_dimension,affected_rows,affected_pct
0,Duplicate application ID (_id),Uniqueness,4,0.8
1,Duplicate valid SSN,Uniqueness,6,1.2
2,Duplicate Full Name + DOB (raw DOB string),Uniqueness,2,0.4


In [31]:
# Distinct records flagged by at least one uniqueness rule (union of checks)
uniqueness_any_flag_mask = (
    df_work["dq_flag_duplicate_id"]
    | df_work["dq_flag_duplicate_ssn"]
    | df_work["dq_flag_duplicate_name_dob"]
)

n_uniqueness_flagged_distinct = int(uniqueness_any_flag_mask.sum())
pct_uniqueness_flagged_distinct = round(n_uniqueness_flagged_distinct / len(df_work) * 100, 2)

print(
    f"Distinct records flagged by at least one uniqueness check: "
    f"{n_uniqueness_flagged_distinct} ({pct_uniqueness_flagged_distinct}%)"
)

# Preview
display(
    df_work.loc[
        uniqueness_any_flag_mask,
        ["_id", "applicant_info_full_name", "applicant_info_date_of_birth"]
    ].sort_values(["_id", "applicant_info_full_name"]).head(20)
)

Distinct records flagged by at least one uniqueness check: 8 (1.59%)


,_id,applicant_info_full_name,applicant_info_date_of_birth
383,app_001,Stephanie Nguyen,1986-05-27
455,app_001,Stephanie Nguyen,NaN
122,app_016,Gary Wilson,1959-12-11
8,app_042,Joseph Lopez,1990-05-04
354,app_042,Joseph Lopez,1990-05-04
92,app_088,Susan Martinez,1986-10-15
16,app_101,Sandra Smith,1997-03-23
499,app_234,Samuel Hill,1976/01/29


The uniqueness assessment identifies both record-level and applicant-level duplication signals in the dataset. Specifically, duplicated application identifiers (`_id`) are observed in **4 records (0.8%)**, duplicated valid SSNs in **6 records (1.2%)**, and duplicated `Full Name + Date of Birth` combinations in **2 records (0.4%)**.

These checks are complementary and overlap in some cases. Therefore, the counts reported by rule are not additive. When combined, a total of **8 distinct records (1.59%)** are flagged by at least one uniqueness rule. Deduplication decisions are deferred to the Remediation section, where stronger identifiers (e.g., `_id`, valid SSN) will be prioritized and flagged cases will be reviewed for traceability.

## Validity

We evaluate **validity** issues by identifying values that are implausible, out of range, or logically impossible given the context of credit applications.

This section focuses on:
- negative values in fields that should be non-negative (e.g., credit history length, savings balance, annual income)
- debt-to-income ratios above expected bounds
- impossible ages derived from `date_of_birth` (e.g., implausibly young or old applicants)

At this stage, we only detect and quantify validity issues. Remediation decisions (e.g., correction, nullification, or exclusion) are deferred to the dedicated Remediation section.

In [33]:
# Validity - numeric plausibility checks


# Analysis-only numeric conversions
analysis_chm = pd.to_numeric(df_work["financials_credit_history_months"], errors="coerce")
analysis_income = pd.to_numeric(df_work["financials_annual_income_canonical"], errors="coerce")
analysis_savings = pd.to_numeric(df_work["financials_savings_balance"], errors="coerce")
analysis_dti = pd.to_numeric(df_work["financials_debt_to_income"], errors="coerce")

In [34]:
neg_chm_mask = analysis_chm < 0

print(f"Negative credit_history_months: {int(neg_chm_mask.sum())} ({pct(int(neg_chm_mask.sum()))}%)")

if neg_chm_mask.any():
    display(
        df_work.loc[
            neg_chm_mask,
            ["_id", "applicant_info_full_name", "financials_credit_history_months"]
        ]
    )

df_work["dq_flag_neg_credit_history_months"] = neg_chm_mask

Negative credit_history_months: 2 (0.4%)


,_id,applicant_info_full_name,financials_credit_history_months
137,app_043,Daniel King,-10
162,app_156,Jessica Green,-3


In [35]:
neg_income_mask = analysis_income < 0

print(f"Negative annual_income (canonical): {int(neg_income_mask.sum())} ({pct(int(neg_income_mask.sum()))}%)")

if neg_income_mask.any():
    display(
        df_work.loc[
            neg_income_mask,
            ["_id", "applicant_info_full_name", "financials_annual_income_canonical"]
        ]
    )

df_work["dq_flag_neg_annual_income"] = neg_income_mask

Negative annual_income (canonical): 0 (0.0%)


In [36]:
neg_savings_mask = analysis_savings < 0

print(f"Negative savings_balance: {int(neg_savings_mask.sum())} ({pct(int(neg_savings_mask.sum()))}%)")

if neg_savings_mask.any():
    display(
        df_work.loc[
            neg_savings_mask,
            ["_id", "applicant_info_full_name", "financials_savings_balance"]
        ]
    )

df_work["dq_flag_neg_savings_balance"] = neg_savings_mask

Negative savings_balance: 1 (0.2%)


,_id,applicant_info_full_name,financials_savings_balance
159,app_290,Stephanie Perez,-5000


In [37]:
dti_over_1_mask = analysis_dti > 1.0

print(f"Debt-to-income ratio > 1.0: {int(dti_over_1_mask.sum())} ({pct(int(dti_over_1_mask.sum()))}%)")

if dti_over_1_mask.any():
    display(
        df_work.loc[
            dti_over_1_mask,
            ["_id", "applicant_info_full_name", "financials_debt_to_income"]
        ]
    )

df_work["dq_flag_dti_over_1"] = dti_over_1_mask

Debt-to-income ratio > 1.0: 1 (0.2%)


,_id,applicant_info_full_name,financials_debt_to_income
316,app_402,Heather Flores,1.85


In [55]:
# Validity - age plausibility check

if "analysis_dob_parsed" not in df_work.columns:
    raise ValueError("analysis_dob_parsed not found. Run the DOB consistency parsing cell before the age validity check.")

today = pd.Timestamp.today().normalize()
df_work["analysis_age_years"] = ((today - df_work["analysis_dob_parsed"]).dt.days / 365.25)

impossible_age_mask = (
    df_work["analysis_age_years"].notna() &
    (
        (df_work["analysis_age_years"] < 18) |
        (df_work["analysis_age_years"] > 120)
    )
)

print(f"Impossible ages (<18 or >120): {int(impossible_age_mask.sum())} ({pct(int(impossible_age_mask.sum()))}%)")

if impossible_age_mask.any():
    age_preview = df_work.loc[
        impossible_age_mask,
        ["_id", "applicant_info_full_name", "applicant_info_date_of_birth", "analysis_dob_parsed", "analysis_age_years"]
    ].copy()
    age_preview["analysis_age_years"] = age_preview["analysis_age_years"].round(1)
    display(age_preview)

df_work["dq_flag_impossible_age"] = impossible_age_mask

Impossible ages (<18 or >120): 0 (0.0%)


In [57]:
# Validity summary by rule (count + percentage)

validity_summary = pd.DataFrame([
    {
        "issue": "Negative credit_history_months",
        "dq_dimension": "Validity",
        "affected_rows": int(df_work["dq_flag_neg_credit_history_months"].sum())
    },
    {
        "issue": "Negative annual_income (canonical)",
        "dq_dimension": "Validity",
        "affected_rows": int(df_work["dq_flag_neg_annual_income"].sum())
    },
    {
        "issue": "Negative savings_balance",
        "dq_dimension": "Validity",
        "affected_rows": int(df_work["dq_flag_neg_savings_balance"].sum())
    },
    {
        "issue": "Debt-to-income ratio > 1.0",
        "dq_dimension": "Validity",
        "affected_rows": int(df_work["dq_flag_dti_over_1"].sum())
    },
    {
        "issue": "Impossible age (<18 or >120)",
        "dq_dimension": "Validity",
        "affected_rows": int(df_work["dq_flag_impossible_age"].sum())
    },
])

validity_summary["affected_pct"] = (validity_summary["affected_rows"] / len(df_work) * 100).round(2)
display(validity_summary)

,issue,dq_dimension,affected_rows,affected_pct
0,Negative credit_history_months,Validity,2,0.4
1,Negative annual_income (canonical),Validity,0,0.0
2,Negative savings_balance,Validity,1,0.2
3,Debt-to-income ratio > 1.0,Validity,1,0.2
4,Impossible age (<18 or >120),Validity,0,0.0


In [58]:
# Distinct records flagged by at least one validity rule
validity_any_flag_mask = (
    df_work["dq_flag_neg_credit_history_months"]
    | df_work["dq_flag_neg_annual_income"]
    | df_work["dq_flag_neg_savings_balance"]
    | df_work["dq_flag_dti_over_1"]
    | df_work["dq_flag_impossible_age"]
)

n_validity_flagged_distinct = int(validity_any_flag_mask.sum())
pct_validity_flagged_distinct = round(n_validity_flagged_distinct / len(df_work) * 100, 2)

print(
    f"Distinct records flagged by at least one validity check: "
    f"{n_validity_flagged_distinct} ({pct_validity_flagged_distinct}%)"
)

display(
    df_work.loc[
        validity_any_flag_mask,
        [
            "_id",
            "applicant_info_full_name",
            "applicant_info_date_of_birth",
            "financials_credit_history_months",
            "financials_annual_income_canonical",
            "financials_savings_balance",
            "financials_debt_to_income"
        ]
    ]
)

Distinct records flagged by at least one validity check: 4 (0.8%)


,_id,applicant_info_full_name,applicant_info_date_of_birth,financials_credit_history_months,financials_annual_income_canonical,financials_savings_balance,financials_debt_to_income
137,app_043,Daniel King,1985-08-17,-10,131000.0,53098,0.06
159,app_290,Stephanie Perez,1990-01-06,39,104000.0,-5000,0.27
162,app_156,Jessica Green,1999-11-28,-3,25000.0,13641,0.21
316,app_402,Heather Flores,1965-03-07,27,88000.0,37281,1.85


The validity assessment identifies a limited number of implausible values in the dataset. Specifically, **2 records (0.4%)** contain negative `credit_history_months`, **1 record (0.2%)** contains a negative `savings_balance`, and **1 record (0.2%)** has a `debt_to_income` ratio above 1.0. No negative values were identified in the canonical annual income field, and no implausible ages (<18 or >120) were detected based on parsed DOB values.

Although validity issues are relatively infrequent, they affect financially relevant variables and may distort downstream risk analysis if left untreated. In total, **4 distinct records (0.8%)** are flagged by at least one validity rule. Remediation decisions are deferred to the Remediation section.